trains, persons, road vehicles, bicycles, signals, signal poles, and catenary poles

# Manual exploration
- Images conditions
  - blurry and dark
    - bright and clear
- Catenary poles might be hard to get when structure is around it (train stations)
- Signals might be far away
- Crowds seem to be used in addition to individual ones(?)
- Overlap possible
- Signal might be on the floor
- Occlussions
- Persons might be standing or sitting


# Load train/val annotations into DataFrames

In [5]:
import matplotlib.pyplot as plt

In [10]:
from pathlib import Path
import json

import pandas as pd

ROOT = Path(Path.cwd()).resolve().parent
ANNOTATIONS_DIR = ROOT / "data" / "annotations" / "objects"
TRAIN_ANN_PATH = ANNOTATIONS_DIR / "annotations_train.json"
VAL_ANN_PATH = ANNOTATIONS_DIR / "annotations_val.json"


def load_coco_annotation_df(annotation_path: Path):
    with annotation_path.open("r", encoding="utf-8") as f:
        coco = json.load(f)

    images_df = pd.DataFrame(coco["images"]).rename(columns={"id": "image_id"})
    annotations_df = pd.DataFrame(coco["annotations"]).rename(columns={"id": "annotation_id"})
    categories_df = pd.DataFrame(coco["categories"]).rename(
        columns={"id": "category_id", "name": "category_name"}
    )

    merged_df = (
        annotations_df.merge(images_df, on="image_id", how="left")
        .merge(categories_df, on="category_id", how="left")
    )

    return images_df, annotations_df, categories_df, merged_df


train_images_df, train_annotations_df, train_categories_df, train_merged_df = load_coco_annotation_df(TRAIN_ANN_PATH)
val_images_df, val_annotations_df, val_categories_df, val_merged_df = load_coco_annotation_df(VAL_ANN_PATH)

print(f"train: {len(train_images_df)} images, {len(train_annotations_df)} annotations")
print(f"val:   {len(val_images_df)} images, {len(val_annotations_df)} annotations")


train: 1500 images, 31431 annotations
val:   500 images, 12335 annotations


## Image sizes

In [ ]:
image_size_counts = (
    train_merged_df[["image_id", "width", "height"]]
    .drop_duplicates()
    .value_counts(["width", "height"])
    .reset_index(name="count")
)
# plot
image_size_counts["size_str"] = (
    image_size_counts["width"].astype(str) + "x" +
    image_size_counts["height"].astype(str)
)

plt.figure(figsize=(8, 4))
plt.bar(image_size_counts["size_str"], image_size_counts["count"])

plt.xlabel("Image size")
plt.ylabel("Number of images")
plt.title("Distribution of image sizes")
plt.xticks(rotation=45)

plt.show()

In [ ]:
image_size_counts = (
    val_images_df[["image_id", "width", "height"]]
    .drop_duplicates()
    .value_counts(["width", "height"])
    .reset_index(name="count")
)
# plot
image_size_counts["size_str"] = (
        image_size_counts["width"].astype(str) + "x" +
        image_size_counts["height"].astype(str)
)

plt.figure(figsize=(8, 4))
plt.bar(image_size_counts["size_str"], image_size_counts["count"])

plt.xlabel("Image size")
plt.ylabel("Number of images")
plt.title("Distribution of image sizes")
plt.xticks(rotation=45)

plt.show()

## Class distribution

In [ ]:
category_counts = train_merged_df["category_name"].value_counts()

import matplotlib.pyplot as plt
axis = category_counts.plot(kind='bar')
axis.set_xlabel("Category")
axis.set_ylabel("Number of images")
axis.set_title("Distribution of categories in training set")

In [ ]:
val_category_counts = val_merged_df["category_name"].value_counts()

axis = val_category_counts.plot(kind='bar')
axis.set_xlabel("Category")
axis.set_ylabel("Number of images")
axis.set_title("Distribution of categories in training set")

## Number of annotations per image


In [ ]:
# Count annotations per image_id in the training set
annotations_per_image = train_annotations_df.groupby("image_id").size()

# Show distribution of number of annotations per image in the training set
counts = annotations_per_image.value_counts().sort_index()
plt.figure()
plt.bar(counts.index, counts.values)
plt.xlabel("Number of annotations per image")
plt.ylabel("Number of images")
plt.title("Distribution of annotations per image")
plt.xticks(range(0, counts.index.max()+1, 10))
plt.show()


In [ ]:
counts.sum() # represents the total number of images in the training set

## Bounding box sizes


In [ ]:
train_merged_df[["bbox_x", "bbox_y", "bbox_w", "bbox_h"]] = pd.DataFrame(
    train_merged_df["bbox"].tolist(),
    index=train_merged_df.index
)
train_merged_df["bbox_area"] = train_merged_df["bbox_w"] * train_merged_df["bbox_h"]
plt.figure()
train_merged_df["bbox_area"].hist(bins=50)
plt.yscale("log")
plt.xlabel("Bounding box area")
plt.ylabel("Frequency")
plt.title("BBox size distribution (absolute values)")
plt.show()

In [ ]:
train_merged_df["relative_area"] = (
    train_merged_df["bbox_area"] / (train_merged_df["width"] * train_merged_df["height"])
)

import numpy as np
values = train_merged_df["relative_area"]

values = np.sort(train_merged_df["relative_area"])

cdf = np.arange(len(values)) / len(values)

plt.figure()
plt.plot(values, cdf)
plt.xscale("log")
plt.xlabel("Relative bbox area")
plt.ylabel("Cumulative probability")
plt.title("CDF of relative bbox sizes")
plt.show()

In [ ]:
quantiles = train_merged_df["relative_area"].quantile([0.5, 0.75, 0.9, 0.95, 0.99])
print(quantiles)

## Width and height distribution

In [ ]:

train_merged_df["bbox_w"] = train_merged_df["bbox"].apply(lambda b: b[2])
train_merged_df["bbox_h"] = train_merged_df["bbox"].apply(lambda b: b[3])

print(train_merged_df[["bbox_w", "bbox_h"]].describe())

# Width plot with log-spaced bins
w = train_merged_df["bbox_w"]
w = w[w > 0]

w_bins = np.logspace(np.log10(w.min()), np.log10(w.max()), 50)

plt.figure()
plt.hist(w, bins=w_bins)
plt.xscale("log")
plt.xlabel("BBox width (pixels)")
plt.ylabel("Frequency")
plt.title("BBox width distribution")

median = w.median()
p75 = w.quantile(0.75)
p25 = w.quantile(0.25)

plt.axvline(median, color='red', linestyle='-', label=f'median ({median:.1f})')
plt.axvline(p75, color='purple', linestyle='--', label=f'75% ({p75:.1f})')
plt.axvline(p25, color='blue', linestyle='--', label=f'25% ({p25:.1f})')

plt.legend(loc="upper right")

plt.show()

# Height plot with log-spaced bins
h = train_merged_df["bbox_h"]
h = h[h > 0]

h_bins = np.logspace(np.log10(h.min()), np.log10(h.max()), 50)

plt.figure()
plt.hist(h, bins=h_bins)
plt.xscale("log")
plt.xlabel("BBox height (pixels)")
plt.ylabel("Frequency")
plt.title("BBox height distribution")

median = h.median()
p75 = h.quantile(0.75)
p25 = h.quantile(0.25)

plt.axvline(median, color='red', linestyle='-', label=f'median ({median:.1f})')
plt.axvline(p75, color='purple', linestyle='--', label=f'75% ({p75:.1f})')
plt.axvline(p25, color='blue', linestyle='--', label=f'25% ({p25:.1f})')

plt.legend(loc="upper right")
plt.show()



## Aspect ratio distribution

In [ ]:
train_merged_df["aspect_ratio"] = train_merged_df["bbox_w"] / train_merged_df["bbox_h"]

aspect = train_merged_df["aspect_ratio"]
aspect = aspect[aspect > 0]  # safety

# log-spaced bins
bins = np.logspace(
    np.log10(aspect.min()),
    np.log10(aspect.max()),
    50
)

# quantiles
q25 = aspect.quantile(0.25)
median = aspect.median()
q75 = aspect.quantile(0.75)

plt.figure(figsize=(7, 4))

plt.hist(aspect, bins=bins)
plt.xscale("log")

# vertical lines
plt.axvline(q25, color='orange', linestyle='--', label=f'25% ({q25:.2f})')
plt.axvline(median, color='red', linestyle='-', label=f'Median ({median:.2f})')
plt.axvline(q75, color='green', linestyle='--', label=f'75% ({q75:.2f})')

plt.xlabel("Aspect ratio (w/h)")
plt.ylabel("Frequency")
plt.title("BBox aspect ratio distribution")

plt.legend()
plt.show()

## Object size by category

In [ ]:
train_merged_df.boxplot(column="relative_area", by="category_name", rot=45)
plt.yscale("log")
plt.title("Relative bbox area by class")
plt.suptitle("")
plt.show()

## Distribution in image by category

In [ ]:
train_merged_df["x_center"] = train_merged_df["bbox"].apply(lambda b: b[0] + b[2] / 2) / train_merged_df["width"]
train_merged_df["y_center"] = train_merged_df["bbox"].apply(lambda b: b[1] + b[3] / 2) / train_merged_df["height"]

classes = train_merged_df["category_name"].unique()
colors = plt.cm.tab10(np.linspace(0, 1, len(classes)))

import seaborn as sns

for cls in train_merged_df["category_name"].unique():
    subset = train_merged_df[train_merged_df["category_name"] == cls]

    plt.figure(figsize=(5, 4))
    sns.kdeplot(
        x=subset["x_center"],
        y=subset["y_center"],
        fill=True,
        cmap="viridis"
    )

    plt.title(f"Spatial distribution: {cls}")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.show()

## Occlusion Evaluation

In [ ]:
anno

## Objects close to borders

In [39]:
train_merged_df["bbox_x"] = train_merged_df["bbox"].apply(lambda b: b[0])
train_merged_df["bbox_y"] = train_merged_df["bbox"].apply(lambda b: b[1])

# .mean() here gives the percentage of boxes that are near the respective border (true values)
near_left = (train_merged_df["bbox_x"] <= 5).mean()
near_top = (train_merged_df["bbox_y"] <= 5).mean()
near_right = ((train_merged_df["bbox_x"] + train_merged_df["bbox_w"]) >= train_merged_df["width"] - 5).mean()
near_bottom = ((train_merged_df["bbox_y"] + train_merged_df["bbox_h"]) >= train_merged_df["height"] - 5).mean()

print(near_left, near_top, near_right, near_bottom)

0.015112468581973211 0.07473513410327384 0.013108078012153607 0.003181572333046992


## Invalid boxes

In [17]:
print((train_merged_df["bbox_w"] <= 0).sum())
print((train_merged_df["bbox_h"] <= 0).sum())

print(((train_merged_df["bbox_x"] + train_merged_df["bbox_w"]) > train_merged_df["width"]).sum())
print(((train_merged_df["bbox_y"] + train_merged_df["bbox_h"]) > train_merged_df["height"]).sum())

0
0
0
0


## Check for NaN or Null values

In [18]:
with TRAIN_ANN_PATH.open("r", encoding="utf-8") as f:
    coco = json.load(f)

images_df = pd.DataFrame(coco["images"]).rename(columns={"id": "image_id"})
annotations_df = pd.DataFrame(coco["annotations"]).rename(columns={"id": "annotation_id"})

train_merged_df = images_df.merge(annotations_df, on="image_id", how="left")
print(f"NaN values:\n{train_merged_df.isna().sum()}")
print(f"\nNull values:\n{train_merged_df.isnull().sum()}")

NaN values:
image_id             0
file_name            0
width                0
height               0
annotation_id        0
category_id          0
bbox                 0
occlusion            0
iscrowd              0
ignore               0
type             28026
dtype: int64

Null values:
image_id             0
file_name            0
width                0
height               0
annotation_id        0
category_id          0
bbox                 0
occlusion            0
iscrowd              0
ignore               0
type             28026
dtype: int64


In [19]:
with VAL_ANN_PATH.open("r", encoding="utf-8") as f:
    coco = json.load(f)

images_df = pd.DataFrame(coco["images"]).rename(columns={"id": "image_id"})
annotations_df = pd.DataFrame(coco["annotations"]).rename(columns={"id": "annotation_id"})

val_merged_df = images_df.merge(annotations_df, on="image_id", how="left")
print(f"NaN values:\n{val_merged_df.isna().sum()}")
print(f"\nNull values:\n{val_merged_df.isnull().sum()}")

NaN values:
image_id             0
file_name            0
width                0
height               0
annotation_id        0
category_id          0
bbox                 0
occlusion            0
iscrowd              0
ignore               0
type             11050
dtype: int64

Null values:
image_id             0
file_name            0
width                0
height               0
annotation_id        0
category_id          0
bbox                 0
occlusion            0
iscrowd              0
ignore               0
type             11050
dtype: int64
